In [10]:
import arcpy
import os
from arcpy.sa import ExtractByMask

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [11]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input GBC and USC raster domains for bathy and slope
rdomain_gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\rdomain_GBC15a_bathy_p26920_v6642"
rdomain_gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\rdomain_GBC15a_slope_p26920_v6642"
rdomain_usc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_bathy_p26920_v6642"
rdomain_usc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_slope_p26920_v6642"

# Output GBC bathy and slope raster domains * with USC overlap removed *
NOOVRLP_rdomain_gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_rdomain_GBC15a_bathy_p26920_v6642"
NOOVRLP_rdomain_gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_rdomain_GBC15a_slope_p26920_v6642"

# ------------

# Input GBC bathy and slope rasters to remove USC overlap area from (using new GBC "nooverlap" raster domains)
gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_bathy_p26920_v6642"
gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_slope_p26920_v6642"

# Output GBC bathy and slope rasters * WITH USC OVERLAP AREA REMOVED * (these rasters will be used in mosaics)
NOOVRLP_gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_bathy_p26920_v6642"
NOOVRLP_gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_slope_p26920_v6642"


In [12]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

# set environment coordinate system: horizontal EPSG:26920 + vertical EPSG:6642 (Pairwise Erase + Extract by Mask tools do NOT carry vertical coordinate system of input(s) through to output layer(s), even if the vert coord system is present in all input layers) - so must manually set coordinate system of output(s) to have both projected/geographic + vertical coordinate systems (in tools' GUIs, this is done by setting "Output Coordinate System" in the 'Environments' tab)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920, 6642)

In [13]:
# ── 1. CREATE GBC RASTER DOMAINS WITH USC OVERLAP REMOVED ────────────────────────────────────────────

# Pairwise Erase (Analysis Tools): Bathymetry
arcpy.analysis.PairwiseErase(
    in_features=rdomain_gbc_bathy,
    # area that will be erased from [in_features] feature class (rdomain_gbc_bathy)
    erase_features=rdomain_usc_bathy,
    out_feature_class=NOOVRLP_rdomain_gbc_bathy
)

# Pairwise Erase (Analysis Tools): Slope
arcpy.analysis.PairwiseErase(
    in_features=rdomain_gbc_slope,
    # area that will be erased from [in_features] feature class (rdomain_slope_bathy)
    erase_features=rdomain_usc_slope,
    out_feature_class=NOOVRLP_rdomain_gbc_slope
)

# print confirmation both no overlap raster domains were created 
print(f"Created: {NOOVRLP_rdomain_gbc_bathy}")
print(f"Created: {NOOVRLP_rdomain_gbc_slope}")

# print coordinate system of each output raster domain
print() # blank line before CRS block
for output in [NOOVRLP_rdomain_gbc_bathy, NOOVRLP_rdomain_gbc_slope]:
    # get spatial reference from output
    sr = arcpy.Describe(output).spatialReference
    # vertical CRS name if present, otherwise flag as missing
    vcs = sr.VCS.name if sr.VCS else "NONE"
    # print horizontal and vertical coordinate systems of each output
    print(f"{os.path.basename(output)} -> \n  Horizontal: {sr.name}\n  Vertical: {vcs}")

Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_rdomain_GBC15a_bathy_p26920_v6642
Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_rdomain_GBC15a_slope_p26920_v6642

nooverlap_rdomain_GBC15a_bathy_p26920_v6642 -> 
  Horizontal: NAD_1983_UTM_Zone_20N
  Vertical: VIVD09_height
nooverlap_rdomain_GBC15a_slope_p26920_v6642 -> 
  Horizontal: NAD_1983_UTM_Zone_20N
  Vertical: VIVD09_height


In [14]:
# ── 2. CREATE GBC RASTERS WITH USC OVERLAP REMOVED ───────────────────────────────────────────────────

# set compression to NONE for output rasters
arcpy.env.compression = "NONE"

# Extract by Mask (Spatial Analyst Tools): Bathymetry
out_bathy = ExtractByMask(
    in_raster=gbc_bathy,
    in_mask_data=NOOVRLP_rdomain_gbc_bathy,
    # 'inside' writes only cells within mask polygon to output raster (essentially crops raster to match exact coverage of mask), all cells outside mask polygon become NoData
    extraction_area="INSIDE"
)
# export output raster to file path specified above
out_bathy.save(NOOVRLP_gbc_bathy)

# Extract by Mask (Spatial Analyst Tools): Slope
out_slope = ExtractByMask(
    in_raster=gbc_slope,
    in_mask_data=NOOVRLP_rdomain_gbc_slope,
    # 'inside' writes only cells within mask polygon to output raster (essentially crops raster to match exact coverage of mask), all cells outside mask polygon become NoData
    extraction_area="INSIDE"
)
# export output raster to file path specified above
out_slope.save(NOOVRLP_gbc_slope)

# print confirmation both rasters (NOOVRLP_gbc_bathy and NOOVRLP_gbc_slope) were created 
print(f"Created: {NOOVRLP_gbc_bathy}")
print(f"Created: {NOOVRLP_gbc_slope}")

# print coordinate system of each output raster domain
print() # blank line before CRS block
for output in [NOOVRLP_gbc_bathy, NOOVRLP_gbc_slope]:
    # get spatial reference from output
    sr = arcpy.Describe(output).spatialReference
    # vertical CRS name if present, otherwise flag as missing
    vcs = sr.VCS.name if sr.VCS else "NONE"
    # print horizontal and vertical coordinate systems of each output
    print(f"{os.path.basename(output)} -> \n  Horizontal: {sr.name}\n  Vertical: {vcs}")

Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_bathy_p26920_v6642
Created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\nooverlap_GBC15a_slope_p26920_v6642

nooverlap_GBC15a_bathy_p26920_v6642 -> 
  Horizontal: NAD_1983_UTM_Zone_20N
  Vertical: VIVD09_height
nooverlap_GBC15a_slope_p26920_v6642 -> 
  Horizontal: NAD_1983_UTM_Zone_20N
  Vertical: VIVD09_height


In [15]:
# reset coordinate system
arcpy.ClearEnvironment("outputCoordinateSystem")

# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")

'CheckedOut'